In [47]:
from dataclasses import dataclass, asdict
from pathlib import Path
from loguru import logger
import numpy as np
import pandas as pd
from SALib.sample import sobol
import psychrolib

from solhycool_visualization.sampling import plot_samples
from phd_visualizations import save_figure

# auto reload modules
%load_ext autoreload
%autoreload 2

psychrolib.SetUnitSystem(psychrolib.SI)
get_Twb = np.vectorize(psychrolib.GetTWetBulbFromRelHum) # Vectorize the function

case_study_id: str = "pilot_plant_200kW"
base_output_path: Path = Path("../results/model_inputs_sampling") / case_study_id
base_output_path.mkdir(parents=True, exist_ok=True)

# Parameters
N: int = 256


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# DC

In [48]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50)
    Tdc_in: tuple[float, float] = (0.1, 50)
    qdc: tuple[float, float] = (6., 24.)
    wdc: tuple[float, float] = (11., 99.18)


In [49]:
# 1. Define problem
output_path = base_output_path / "dc.csv"
mip = ModelInputsRange()

var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Tdc_out']
}


In [50]:
# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Tamb
df = df[df["Tdc_in"] < df["Tamb"]]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")


2025-04-17 06:45:36.082 | INFO     | __main__:<module>:11 - After filtering, removed 1292 rows, from 2560 to 1268


,Tamb,Tdc_in,qdc,wdc
0,44.664323,35.577624,13.771790,58.212372
2,44.664323,12.510722,13.771790,58.212372
3,44.664323,35.577624,20.153848,58.212372
4,44.664323,35.577624,13.771790,26.110962
5,44.664323,12.510722,20.153848,26.110962
...,...,...,...,...
2552,44.776001,41.007625,16.396529,28.373795
2553,44.776001,3.036226,9.111720,28.373795
2554,44.776001,3.036226,16.396529,15.651460
2555,44.776001,41.007625,9.111720,15.651460


2025-04-17 06:45:36.098 | INFO     | __main__:<module>:14 - Samples saved at ../results/model_inputs_sampling/pilot_plant_200kW/dc.csv


In [51]:
fig = plot_samples(df, var_ids, Ncols=len(var_ids))
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_dc", formats=["png"])

fig


2025-04-17 06:45:36.892 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/model_inputs_sampling/pilot_plant_200kW')]/viz_samples_dc.png


In [52]:
# 4. Visualize new data-based models performance


# WCT

In [53]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50)
    HR: tuple[float, float] = (10.33, 89.25)
    Twct_in: tuple[float, float] = (0.1, 50)
    qwct: tuple[float, float] = (6., 24.)
    wwct: tuple[float, float] = (0.0, 93.4161)


In [54]:
# 1. Define problem

output_path = base_output_path / "wct.csv"
mip = ModelInputsRange()
var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Twct_out']
}


In [55]:
# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Twb
df = df[ df["Twct_in"] > get_Twb(df["Tamb"], df["HR"]/100, np.full(len(df), 101325)) ]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")


2025-04-17 06:45:37.131 | INFO     | __main__:<module>:11 - After filtering, removed 1095 rows, from 3072 to 1977


,Tamb,HR,Twct_in,qwct,wwct
0,22.145903,14.870115,46.687686,17.531380,47.553199
1,2.678613,14.870115,46.687686,17.531380,47.553199
2,22.145903,11.690021,46.687686,17.531380,47.553199
4,22.145903,14.870115,46.687686,15.418363,47.553199
5,22.145903,14.870115,46.687686,17.531380,29.997230
...,...,...,...,...,...
3055,16.378204,42.788074,49.910703,9.259382,60.828721
3056,16.378204,37.962086,31.310028,9.259382,60.828721
3057,16.378204,37.962086,49.910703,19.219575,60.828721
3058,16.378204,37.962086,49.910703,9.259382,56.259326


2025-04-17 06:45:37.155 | INFO     | __main__:<module>:14 - Samples saved at ../results/model_inputs_sampling/pilot_plant_200kW/wct.csv


In [56]:
fig = plot_samples(df, var_ids, Ncols=len(var_ids))
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_wct", formats=["png"])

fig


2025-04-17 06:45:38.455 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/model_inputs_sampling/pilot_plant_200kW')]/viz_samples_wct.png
